In [ ]:
import glob
import json
from pathlib import Path

import pandas as pd
import seaborn as sns
sns.set()

from hipe_ocrepair_scorer.cli import (
    load_jsonl,
    load_schema,
    merge_reference_hypothesis,
    find_reference_files,
    find_hypothesis_file,
    validate_jsonl_record
)
from hipe_ocrepair_scorer.ocrepair_eval import Evaluation

In [ ]:
REF_DIR = Path("../data/processed/train_ref/")
HYP_DIR = Path("../data/processed/train_hyp_as_postcorr/")

In [ ]:
# Only requires document_metadata and ocr_hypothesis, doesn't have to include ground_truth or ocr_postcorrection_output
schema = load_schema("../HIPE-OCRepair-scorer/data/schema/hipe-ocrepair.schema.json")  

In [ ]:
ocr_postcorrection_output = {
    "transcription_unit": "None",
    "ocr_postcorrection_system": "None",
    "num_tokens": -1,
    "num_chars": -1,
    "quality_report": {
        "ocr_quality_score": -1,
        "cer": -1, "wer": -1,
        "nb_char_substitutions": -1,
        "nb_char_deletions": -1,
        "nb_char_insertions": -1}
}

In [ ]:
for p in [p for p in HYP_DIR.glob("*.jsonl")]:
    print(p)
    with open(p, "r") as f:
        json_lines = f.readlines()
        records = [json.loads(l) for l in json_lines]
        
    for i, record in enumerate(records):
        validate_jsonl_record(record=record, schema=schema, filepath=p, line_num=i)
        print(record.keys())
        if "ocr_postcorrection_output" not in record:
            record["ocr_postcorrection_output"] = ocr_postcorrection_output
        
        record["ocr_postcorrection_output"]["transcription_unit"] = record["ocr_hypothesis"]["transcription_unit"]

    with open(p, "w") as f:
        [f.write(json.dumps(record) + "\n") for record in records]

In [ ]:
def score_folder_pair(ref_dir, hyp_dir):
    """Evaluate all matching files across reference and hypothesis directories."""
    print("=" * 60)
    print("Example 2: Folder pair")
    print("=" * 60)
    print(f"  Reference dir:  {ref_dir}")
    print(f"  Hypothesis dir: {hyp_dir}")
    print()

    ref_files = find_reference_files(ref_dir)
    all_merged = []

    for ref_path in ref_files:
        hyp_path = find_hypothesis_file(hyp_dir, ref_path)
        if hyp_path is None:
            print(f"  [SKIP] No hypothesis file for {ref_path.name}")
            continue

        print(f"  {ref_path.name} <-> {hyp_path.name}")
        ref_records = load_jsonl(ref_path)
        hyp_records = load_jsonl(hyp_path)
        merged = merge_reference_hypothesis(ref_records, hyp_records)
        all_merged.extend(merged)

    print(f"\n  Total: {len(all_merged)} documents")

    evaluation = Evaluation(all_merged)
    results = evaluation.score_over_datasets(normalize=True)

    return results

In [ ]:
results = score_folder_pair(REF_DIR, HYP_DIR)

In [ ]:
results_df = pd.DataFrame(results["fold_scores"]).T
results_df["cmer_micro_5ci"] = results_df["cmer_micro"].apply(lambda x: x[1])
results_df["cmer_micro_95ci"] = results_df["cmer_micro"].apply(lambda x: x[2])
results_df["cmer_micro"] = results_df["cmer_micro"].apply(lambda x: x[0])

results_df["wmer_micro_5ci"] = results_df["wmer_micro"].apply(lambda x: x[1])
results_df["wmer_micro_95ci"] = results_df["wmer_micro"].apply(lambda x: x[2])
results_df["wmer_micro"] = results_df["wmer_micro"].apply(lambda x: x[0])

results_df["cmer_macro_5ci"] = results_df["cmer_macro"].apply(lambda x: x[1])
results_df["cmer_macro_95ci"] = results_df["cmer_macro"].apply(lambda x: x[2])
results_df["cmer_macro"] = results_df["cmer_macro"].apply(lambda x: x[0])

results_df["wmer_macro_5ci"] = results_df["wmer_macro"].apply(lambda x: x[1])
results_df["wmer_macro_95ci"] = results_df["wmer_macro"].apply(lambda x: x[2])
results_df["wmer_macro"] = results_df["wmer_macro"].apply(lambda x: x[0])

results_df

In [ ]:
long_results_df = results_df.iloc[:, :4].stack().rename("score").reset_index().rename(columns={"level_0": "dataset", "level_1": "metric"})

In [ ]:
g = sns.FacetGrid(long_results_df, col="metric")
g.map(sns.scatterplot, "dataset", "score")

In [ ]:
[ax.tick_params(rotation=45) for ax in g.axes[0, :]]

In [ ]:
!mkdir ..\reports\figures\score_vis

In [ ]:
g.figure.savefig("../reports/figures/score_vis/score_vis_demo.png", bbox_inches="tight")